# Hard-Level Pandas Practice Questions with Detailed Solutions

This notebook contains **2 hard-level Pandas practice questions**. Each question is written in detail first, then its complete solution is given step by step before the next question begins.

## Functions and operations covered

`head()`, `tail()`, `describe()`, `corr()`, `sort_values()`, `value_counts()`, `groupby()`, `merge()`, `concat()`, `filter()`, `query()`, `loc[]`, `iloc[]`, `duplicated()`, `drop_duplicates()`, `fillna()`, `isin()`, `rename()`, `reset_index()`, and multiple aggregations.

## Common Setup

Run this setup cell first. The CSV datasets are embedded inside the notebook, so you do not need to download or create separate CSV files.

In [1]:
import pandas as pd
from io import StringIO

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Question 1: Hard-Level E-Commerce Sales, Customer, and Returns Analysis

## Business case

A company sells products through online and store channels across four regions. The management team wants a detailed analysis of orders, customers, discounts, delivery status, and returned products. The dataset intentionally contains duplicate rows and missing values, so data cleaning is required before analysis.

## Embedded CSV files

You are given three CSV datasets inside the notebook:

- `orders_csv`: order transaction details
- `customers_csv`: customer demographic and loyalty details
- `returns_csv`: returned order details

## Detailed tasks

1. Read `orders_csv`, `customers_csv`, and `returns_csv` into three DataFrames.
2. Display the shape of each DataFrame.
3. Display the first 6 records of the orders DataFrame using `head()`.
4. Display the last 4 records of the orders DataFrame using `tail()`.
5. Display the statistical summary of numeric columns using `describe()`.
6. Find correlation among numeric columns using `corr()`.
7. Rename `OrderAmount` as `Revenue` and `DiscountPct` as `DiscountRate` using `rename()`.
8. Check whether duplicate order rows exist using `duplicated()`.
9. Remove duplicate order rows using `drop_duplicates()`.
10. Fill missing `DiscountRate` values with `0` using `fillna()`.
11. Fill missing `CustomerRating` values with the average customer rating using `fillna()`.
12. Select only important columns using `filter()`.
13. Count orders by `Channel`, `Region`, and `PaymentMethod` using `value_counts()`.
14. Sort orders by `Revenue` in descending order using `sort_values()`.
15. Use `loc[]` to display only delivered orders where revenue is at least 40000.
16. Use `iloc[]` to display rows 3 to 8 and the first 7 columns.
17. Use `query()` to display orders where shipping cost is at least 700 and profit is greater than 5000.
18. Use `isin()` to select orders from `West` or `North` region and from `Electronics` or `Furniture` category.
19. Merge cleaned orders with customers using `CustomerID`.
20. Merge the result with returns using `OrderID`.
21. Use `groupby()` with multiple aggregations to calculate total revenue, average revenue, total profit, average profit, total quantity, average rating, order count, and returned order count by `Region` and `Category`.
22. Use `reset_index()` after groupby.
23. Concatenate three new orders with the cleaned orders DataFrame using `concat()`.
24. Create a final summary by `Region` and `Channel` showing order count, revenue, profit, average discount, and average rating.

## Expected difficulty level

This is a hard-level question because it combines data cleaning, duplicate handling, missing value treatment, conditional filtering, multi-table merging, row concatenation, and grouped business reporting.

# Solution 1: E-Commerce Analysis

The following cells solve Question 1 step by step. Run them in order.

## Step 1: Read the Embedded CSV Datasets

The CSV strings are loaded with `StringIO`, which allows Pandas to read text data as if it were a normal CSV file.

In [2]:
orders_csv = """OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,OrderAmount,DiscountPct,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
1001,C001,2026-01-05,West,Electronics,Laptop,1,72000,0.10,900,8500,UPI,Online,Delivered,4.6
1002,C002,2026-01-06,North,Furniture,Office Chair,3,18000,,1200,2400,Credit Card,Store,Delivered,4.1
1003,C003,2026-01-07,South,Electronics,Headphones,5,12500,0.05,450,1800,Debit Card,Online,Delivered,4.4
1004,C004,2026-01-08,East,Clothing,Jacket,4,9600,0.15,300,1100,UPI,Online,Returned,
1005,C005,2026-01-09,West,Grocery,Organic Rice,10,8500,0.02,250,900,Cash,Store,Delivered,3.9
1006,C006,2026-01-10,North,Electronics,Smartphone,2,68000,0.08,700,7600,Credit Card,Online,Delivered,4.8
1007,C007,2026-01-11,South,Furniture,Study Table,2,30000,,1500,4200,UPI,Store,Delayed,4.0
1008,C008,2026-01-12,East,Clothing,Shoes,6,15000,0.12,500,2100,Debit Card,Online,Delivered,4.3
1009,C009,2026-01-13,West,Electronics,Tablet,3,54000,0.07,650,6900,Credit Card,Online,Delivered,4.7
1010,C010,2026-01-14,North,Grocery,Protein Pack,8,14400,0.03,350,1600,UPI,Store,Cancelled,3.8
1011,C011,2026-01-15,South,Clothing,T-Shirt,12,7200,0.20,280,950,Cash,Online,Delivered,4.2
1012,C012,2026-01-16,East,Electronics,Monitor,4,48000,0.09,800,6100,Credit Card,Store,Delivered,4.5
1013,C013,2026-01-17,West,Furniture,Bookshelf,2,22000,0.06,1300,3300,Debit Card,Online,Returned,3.7
1014,C014,2026-01-18,North,Clothing,Jeans,7,13300,,420,1750,UPI,Online,Delivered,4.0
1015,C015,2026-01-19,South,Grocery,Coffee Beans,9,11700,0.04,320,1400,Cash,Store,Delivered,
1016,C016,2026-01-20,East,Furniture,Sofa,1,42000,0.11,2000,5200,Credit Card,Store,Delayed,4.1
1017,C017,2026-01-21,West,Electronics,Camera,2,56000,0.10,750,7200,UPI,Online,Delivered,4.9
1018,C018,2026-01-22,North,Grocery,Dry Fruits,6,15600,0.05,360,1900,Debit Card,Online,Delivered,4.4
1019,C019,2026-01-23,South,Furniture,Bed Frame,1,38000,0.08,1700,4600,Credit Card,Store,Returned,3.9
1020,C020,2026-01-24,East,Clothing,Formal Shirt,10,16000,0.18,520,2300,UPI,Online,Delivered,4.6
1017,C017,2026-01-21,West,Electronics,Camera,2,56000,0.10,750,7200,UPI,Online,Delivered,4.9
"""

customers_csv = """CustomerID,CustomerName,AgeGroup,LoyaltyTier,City
C001,Aarav,26-35,Gold,Ahmedabad
C002,Diya,36-45,Silver,Delhi
C003,Kabir,18-25,Bronze,Chennai
C004,Meera,26-35,Gold,Kolkata
C005,Rohan,46-55,Silver,Surat
C006,Anaya,26-35,Platinum,Jaipur
C007,Vivaan,36-45,Gold,Bengaluru
C008,Isha,18-25,Bronze,Bhubaneswar
C009,Arjun,26-35,Platinum,Mumbai
C010,Sara,36-45,Silver,Lucknow
C011,Dev,18-25,Bronze,Hyderabad
C012,Tara,26-35,Gold,Patna
C013,Nikhil,46-55,Silver,Pune
C014,Kavya,26-35,Gold,Chandigarh
C015,Manav,36-45,Bronze,Coimbatore
C016,Reva,46-55,Silver,Ranchi
C017,Om,26-35,Platinum,Vadodara
C018,Siya,18-25,Gold,Noida
C019,Yash,36-45,Silver,Mysuru
C020,Aditi,26-35,Gold,Guwahati
"""

returns_csv = """OrderID,ReturnReason,ReturnDays
1004,Size Issue,5
1013,Damaged Product,3
1019,Changed Mind,7
"""

orders = pd.read_csv(StringIO(orders_csv), parse_dates=["OrderDate"])
customers = pd.read_csv(StringIO(customers_csv))
returns = pd.read_csv(StringIO(returns_csv))

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Returns shape:", returns.shape)

Orders shape: (21, 15)
Customers shape: (20, 5)
Returns shape: (3, 3)


## Step 2: Display First and Last Records

`head()` quickly previews the beginning of the dataset, while `tail()` checks the ending rows.

In [3]:
display(orders.head(6))
display(orders.tail(4))

,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,OrderAmount,DiscountPct,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
0,1001,C001,2026-01-05,West,Electronics,Laptop,1,72000,0.10,900,8500,UPI,Online,Delivered,4.6
1,1002,C002,2026-01-06,North,Furniture,Office Chair,3,18000,NaN,1200,2400,Credit Card,Store,Delivered,4.1
2,1003,C003,2026-01-07,South,Electronics,Headphones,5,12500,0.05,450,1800,Debit Card,Online,Delivered,4.4
3,1004,C004,2026-01-08,East,Clothing,Jacket,4,9600,0.15,300,1100,UPI,Online,Returned,NaN
4,1005,C005,2026-01-09,West,Grocery,Organic Rice,10,8500,0.02,250,900,Cash,Store,Delivered,3.9
5,1006,C006,2026-01-10,North,Electronics,Smartphone,2,68000,0.08,700,7600,Credit Card,Online,Delivered,4.8


,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,OrderAmount,DiscountPct,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
17,1018,C018,2026-01-22,North,Grocery,Dry Fruits,6,15600,0.05,360,1900,Debit Card,Online,Delivered,4.4
18,1019,C019,2026-01-23,South,Furniture,Bed Frame,1,38000,0.08,1700,4600,Credit Card,Store,Returned,3.9
19,1020,C020,2026-01-24,East,Clothing,Formal Shirt,10,16000,0.18,520,2300,UPI,Online,Delivered,4.6
20,1017,C017,2026-01-21,West,Electronics,Camera,2,56000,0.10,750,7200,UPI,Online,Delivered,4.9


## Step 3: Statistical Summary

`describe()` gives count, mean, standard deviation, minimum, quartiles, and maximum for numeric columns.

In [4]:
orders.describe()

,OrderID,OrderDate,Quantity,OrderAmount,DiscountPct,ShippingCost,Profit,CustomerRating
count,21.000000,21,21.000000,21.000000,18.000000,21.000000,21.000000,19.000000
mean,1010.809524,2026-01-14 19:25:42.857142784,4.761905,29895.238095,0.090556,761.904762,3761.904762,4.305263
min,1001.000000,2026-01-05 00:00:00,1.000000,7200.000000,0.020000,250.000000,900.000000,3.700000
25%,1006.000000,2026-01-10 00:00:00,2.000000,13300.000000,0.052500,360.000000,1750.000000,4.000000
50%,1011.000000,2026-01-15 00:00:00,4.000000,18000.000000,0.085000,650.000000,2400.000000,4.300000
75%,1016.000000,2026-01-20 00:00:00,7.000000,48000.000000,0.107500,900.000000,6100.000000,4.600000
max,1020.000000,2026-01-24 00:00:00,12.000000,72000.000000,0.200000,2000.000000,8500.000000,4.900000
std,5.938174,NaN,3.404479,21290.572472,0.049166,502.579536,2557.973063,0.377821


## Step 4: Correlation Analysis

`corr()` shows relationships between numeric columns such as revenue, quantity, shipping cost, profit, and customer rating.

In [5]:
orders.corr(numeric_only=True)

,OrderID,Quantity,OrderAmount,DiscountPct,ShippingCost,Profit,CustomerRating
OrderID,1.000000,0.094101,0.003908,0.138517,0.153089,0.036859,0.081447
Quantity,0.094101,1.000000,-0.714664,0.148133,-0.717422,-0.735715,-0.212110
OrderAmount,0.003908,-0.714664,1.000000,-0.018333,0.391713,0.993433,0.643818
DiscountPct,0.138517,0.148133,-0.018333,1.000000,0.010992,-0.000774,0.303151
ShippingCost,0.153089,-0.717422,0.391713,0.010992,1.000000,0.425430,-0.256820
Profit,0.036859,-0.735715,0.993433,-0.000774,0.425430,1.000000,0.649044
CustomerRating,0.081447,-0.212110,0.643818,0.303151,-0.256820,0.649044,1.000000


## Step 5: Rename Columns

Column names are renamed to make the business meaning clearer.

In [6]:
orders_renamed = orders.rename(columns={"OrderAmount": "Revenue", "DiscountPct": "DiscountRate"})
orders_renamed.head()

,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,Revenue,DiscountRate,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
0,1001,C001,2026-01-05,West,Electronics,Laptop,1,72000,0.10,900,8500,UPI,Online,Delivered,4.6
1,1002,C002,2026-01-06,North,Furniture,Office Chair,3,18000,NaN,1200,2400,Credit Card,Store,Delivered,4.1
2,1003,C003,2026-01-07,South,Electronics,Headphones,5,12500,0.05,450,1800,Debit Card,Online,Delivered,4.4
3,1004,C004,2026-01-08,East,Clothing,Jacket,4,9600,0.15,300,1100,UPI,Online,Returned,NaN
4,1005,C005,2026-01-09,West,Grocery,Organic Rice,10,8500,0.02,250,900,Cash,Store,Delivered,3.9


## Step 6: Detect Duplicate Rows

`duplicated()` returns `True` for repeated rows.

In [7]:
orders_renamed.duplicated().value_counts()

False    20
True      1
Name: count, dtype: int64

## Step 7: Remove Duplicate Rows

`drop_duplicates()` keeps one copy of each duplicate row.

In [8]:
orders_clean = orders_renamed.drop_duplicates().copy()
print("Before removing duplicates:", orders_renamed.shape)
print("After removing duplicates:", orders_clean.shape)

Before removing duplicates: (21, 15)
After removing duplicates: (20, 15)


In [9]:
orders_clean['CustomerRating'].mean()

4.272222222222222

## Step 8: Fill Missing Values

Missing discounts are treated as no discount. Missing customer ratings are filled with the average rating.

In [10]:
orders_clean["DiscountRate"] = orders_clean["DiscountRate"].fillna(0)
orders_clean["CustomerRating"] = orders_clean["CustomerRating"].fillna(orders_clean["CustomerRating"].mean())
orders_clean[["OrderID", "DiscountRate", "CustomerRating"]].head(10)

,OrderID,DiscountRate,CustomerRating
0,1001,0.10,4.600000
1,1002,0.00,4.100000
2,1003,0.05,4.400000
3,1004,0.15,4.272222
4,1005,0.02,3.900000
5,1006,0.08,4.800000
6,1007,0.00,4.000000
7,1008,0.12,4.300000
8,1009,0.07,4.700000
9,1010,0.03,3.800000


## Step 9: Filter Important Columns

`filter()` is used to select only required columns by name.

In [11]:
orders_clean.filter(items=["OrderID", "CustomerID", "Region", "Category", "Revenue", "Profit", "CustomerRating"]).head()

,OrderID,CustomerID,Region,Category,Revenue,Profit,CustomerRating
0,1001,C001,West,Electronics,72000,8500,4.600000
1,1002,C002,North,Furniture,18000,2400,4.100000
2,1003,C003,South,Electronics,12500,1800,4.400000
3,1004,C004,East,Clothing,9600,1100,4.272222
4,1005,C005,West,Grocery,8500,900,3.900000


## Step 10: Count Category Frequencies

`value_counts()` helps understand distribution across channels, regions, and payment methods.

In [12]:
display(orders_clean["Channel"].value_counts())
display(orders_clean["Region"].value_counts())
display(orders_clean["PaymentMethod"].value_counts())

Channel
Online    12
Store      8
Name: count, dtype: int64

Region
West     5
North    5
South    5
East     5
Name: count, dtype: int64

PaymentMethod
UPI            7
Credit Card    6
Debit Card     4
Cash           3
Name: count, dtype: int64

## Step 11: Sort by Revenue

`sort_values()` is used to find the highest revenue orders.

In [13]:
orders_clean.sort_values(by="Revenue", ascending=False).head(8)

,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,Revenue,DiscountRate,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
0,1001,C001,2026-01-05,West,Electronics,Laptop,1,72000,0.10,900,8500,UPI,Online,Delivered,4.6
5,1006,C006,2026-01-10,North,Electronics,Smartphone,2,68000,0.08,700,7600,Credit Card,Online,Delivered,4.8
16,1017,C017,2026-01-21,West,Electronics,Camera,2,56000,0.10,750,7200,UPI,Online,Delivered,4.9
8,1009,C009,2026-01-13,West,Electronics,Tablet,3,54000,0.07,650,6900,Credit Card,Online,Delivered,4.7
11,1012,C012,2026-01-16,East,Electronics,Monitor,4,48000,0.09,800,6100,Credit Card,Store,Delivered,4.5
15,1016,C016,2026-01-20,East,Furniture,Sofa,1,42000,0.11,2000,5200,Credit Card,Store,Delayed,4.1
18,1019,C019,2026-01-23,South,Furniture,Bed Frame,1,38000,0.08,1700,4600,Credit Card,Store,Returned,3.9
6,1007,C007,2026-01-11,South,Furniture,Study Table,2,30000,0.00,1500,4200,UPI,Store,Delayed,4.0


## Step 12: Use loc for Conditional Selection

`loc[]` selects rows by condition and columns by label.

In [14]:
orders_clean.loc[(orders_clean["Revenue"] >= 40000) & (orders_clean["DeliveryStatus"] == "Delivered"), ["OrderID", "Region", "Category", "Product", "Revenue", "Profit", "DeliveryStatus"]]

,OrderID,Region,Category,Product,Revenue,Profit,DeliveryStatus
0,1001,West,Electronics,Laptop,72000,8500,Delivered
5,1006,North,Electronics,Smartphone,68000,7600,Delivered
8,1009,West,Electronics,Tablet,54000,6900,Delivered
11,1012,East,Electronics,Monitor,48000,6100,Delivered
16,1017,West,Electronics,Camera,56000,7200,Delivered


## Step 13: Use iloc for Position-Based Selection

`iloc[]` selects rows and columns by integer position.

In [15]:
orders_clean.iloc[2:8, 0:7]

,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity
2,1003,C003,2026-01-07,South,Electronics,Headphones,5
3,1004,C004,2026-01-08,East,Clothing,Jacket,4
4,1005,C005,2026-01-09,West,Grocery,Organic Rice,10
5,1006,C006,2026-01-10,North,Electronics,Smartphone,2
6,1007,C007,2026-01-11,South,Furniture,Study Table,2
7,1008,C008,2026-01-12,East,Clothing,Shoes,6


## Step 14: Use query for Business Conditions

`query()` makes complex conditions easier to read.

In [16]:
orders_clean.query("ShippingCost >= 700 and Profit > 5000")

,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,Revenue,DiscountRate,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
0,1001,C001,2026-01-05,West,Electronics,Laptop,1,72000,0.10,900,8500,UPI,Online,Delivered,4.6
5,1006,C006,2026-01-10,North,Electronics,Smartphone,2,68000,0.08,700,7600,Credit Card,Online,Delivered,4.8
11,1012,C012,2026-01-16,East,Electronics,Monitor,4,48000,0.09,800,6100,Credit Card,Store,Delivered,4.5
15,1016,C016,2026-01-20,East,Furniture,Sofa,1,42000,0.11,2000,5200,Credit Card,Store,Delayed,4.1
16,1017,C017,2026-01-21,West,Electronics,Camera,2,56000,0.10,750,7200,UPI,Online,Delivered,4.9


## Step 15: Use isin for Multiple Allowed Values

`isin()` is useful when a column can match any value from a list.

In [17]:
orders_clean[orders_clean["Region"].isin(["West", "North"]) & orders_clean["Category"].isin(["Electronics", "Furniture"])][["OrderID", "Region", "Category", "Product", "Revenue", "Profit"]]

,OrderID,Region,Category,Product,Revenue,Profit
0,1001,West,Electronics,Laptop,72000,8500
1,1002,North,Furniture,Office Chair,18000,2400
5,1006,North,Electronics,Smartphone,68000,7600
8,1009,West,Electronics,Tablet,54000,6900
12,1013,West,Furniture,Bookshelf,22000,3300
16,1017,West,Electronics,Camera,56000,7200


## Step 16: Merge Related Tables

The orders table is merged with customers and returns to create one enriched dataset.

In [18]:
order_customer = orders_clean.merge(customers, on="CustomerID", how="left")
full_orders = order_customer.merge(returns, on="OrderID", how="left")
full_orders[["OrderID", "CustomerName", "LoyaltyTier", "Region", "Category", "Revenue", "DeliveryStatus", "ReturnReason"]].head(10)

,OrderID,CustomerName,LoyaltyTier,Region,Category,Revenue,DeliveryStatus,ReturnReason
0,1001,Aarav,Gold,West,Electronics,72000,Delivered,NaN
1,1002,Diya,Silver,North,Furniture,18000,Delivered,NaN
2,1003,Kabir,Bronze,South,Electronics,12500,Delivered,NaN
3,1004,Meera,Gold,East,Clothing,9600,Returned,Size Issue
4,1005,Rohan,Silver,West,Grocery,8500,Delivered,NaN
5,1006,Anaya,Platinum,North,Electronics,68000,Delivered,NaN
6,1007,Vivaan,Gold,South,Furniture,30000,Delayed,NaN
7,1008,Isha,Bronze,East,Clothing,15000,Delivered,NaN
8,1009,Arjun,Platinum,West,Electronics,54000,Delivered,NaN
9,1010,Sara,Silver,North,Grocery,14400,Cancelled,NaN


## Step 17: Groupby with Multiple Aggregations

This grouped report calculates several business metrics together for each region and category.

In [19]:
region_category_summary = (
    full_orders
    .groupby(["Region", "Category"])
    .agg(
        total_revenue=("Revenue", "sum"),
        avg_revenue=("Revenue", "mean"),
        total_profit=("Profit", "sum"),
        avg_profit=("Profit", "mean"),
        total_quantity=("Quantity", "sum"),
        avg_rating=("CustomerRating", "mean"),
        order_count=("OrderID", "count"),
        returned_orders=("ReturnReason", lambda x: x.notna().sum())
    )
    .reset_index()
)
region_category_summary.sort_values(by="total_revenue", ascending=False)

,Region,Category,total_revenue,avg_revenue,total_profit,avg_profit,total_quantity,avg_rating,order_count,returned_orders
11,West,Electronics,182000,60666.666667,22600,7533.333333,6,4.733333,3,0
4,North,Electronics,68000,68000.000000,7600,7600.000000,2,4.800000,1,0
9,South,Furniture,68000,34000.000000,8800,4400.000000,3,3.950000,2,1
1,East,Electronics,48000,48000.000000,6100,6100.000000,4,4.500000,1,0
2,East,Furniture,42000,42000.000000,5200,5200.000000,1,4.100000,1,0
0,East,Clothing,40600,13533.333333,5500,1833.333333,20,4.390741,3,1
6,North,Grocery,30000,15000.000000,3500,1750.000000,14,4.100000,2,0
12,West,Furniture,22000,22000.000000,3300,3300.000000,2,3.700000,1,1
5,North,Furniture,18000,18000.000000,2400,2400.000000,3,4.100000,1,0
3,North,Clothing,13300,13300.000000,1750,1750.000000,7,4.000000,1,0


## Step 18: Concatenate New Orders

`concat()` adds new order rows to the cleaned orders DataFrame.

In [20]:
new_orders_csv = """OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,Revenue,DiscountRate,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
1021,C006,2026-01-25,North,Electronics,Smartwatch,4,36000,0.06,600,5100,UPI,Online,Delivered,4.5
1022,C014,2026-01-26,West,Furniture,Recliner,1,28000,0.07,1400,3900,Credit Card,Store,Delivered,4.2
1023,C003,2026-01-27,South,Clothing,Hoodie,8,18400,0.10,550,2600,Debit Card,Online,Delivered,4.3
"""
new_orders = pd.read_csv(StringIO(new_orders_csv), parse_dates=["OrderDate"])
orders_final = pd.concat([orders_clean, new_orders], ignore_index=True)
print("Final orders shape:", orders_final.shape)
orders_final.tail(6)

Final orders shape: (23, 15)


,OrderID,CustomerID,OrderDate,Region,Category,Product,Quantity,Revenue,DiscountRate,ShippingCost,Profit,PaymentMethod,Channel,DeliveryStatus,CustomerRating
17,1018,C018,2026-01-22,North,Grocery,Dry Fruits,6,15600,0.05,360,1900,Debit Card,Online,Delivered,4.4
18,1019,C019,2026-01-23,South,Furniture,Bed Frame,1,38000,0.08,1700,4600,Credit Card,Store,Returned,3.9
19,1020,C020,2026-01-24,East,Clothing,Formal Shirt,10,16000,0.18,520,2300,UPI,Online,Delivered,4.6
20,1021,C006,2026-01-25,North,Electronics,Smartwatch,4,36000,0.06,600,5100,UPI,Online,Delivered,4.5
21,1022,C014,2026-01-26,West,Furniture,Recliner,1,28000,0.07,1400,3900,Credit Card,Store,Delivered,4.2
22,1023,C003,2026-01-27,South,Clothing,Hoodie,8,18400,0.10,550,2600,Debit Card,Online,Delivered,4.3


## Step 19: Final Region and Channel Summary

The final summary shows sales performance after adding the new order records.

In [21]:
final_order_summary = (
    orders_final
    .groupby(["Region", "Channel"])
    .agg(
        orders=("OrderID", "count"),
        revenue=("Revenue", "sum"),
        profit=("Profit", "sum"),
        avg_discount=("DiscountRate", "mean"),
        avg_rating=("CustomerRating", "mean")
    )
    .reset_index()
    .sort_values(by=["revenue", "profit"], ascending=False)
)
final_order_summary

,Region,Channel,orders,revenue,profit,avg_discount,avg_rating
6,West,Online,4,204000,25900,0.082500,4.475000
2,North,Online,4,132900,16350,0.047500,4.425000
1,East,Store,2,90000,11300,0.100000,4.300000
5,South,Store,3,79700,10200,0.040000,4.057407
0,East,Online,3,40600,5500,0.150000,4.390741
4,South,Online,3,38100,5350,0.116667,4.300000
7,West,Store,2,36500,4800,0.045000,4.050000
3,North,Store,2,32400,4000,0.015000,3.950000


# Question 2: Hard-Level Employee Performance, Salary, and Training Analysis

## Business case

A company wants to analyze employee salary, performance, attendance, training completion, work mode, and department-level payroll cost. The dataset contains duplicate employee rows and missing salary/rating values, so cleaning is required before analysis.

## Embedded CSV files

You are given three CSV datasets inside the notebook:

- `employees_csv`: employee salary and performance data
- `departments_csv`: department manager, budget, and bonus-rate data
- `training_csv`: employee training completion data

## Detailed tasks

1. Read `employees_csv`, `departments_csv`, and `training_csv` into DataFrames.
2. Display the shape of each DataFrame.
3. Display first 7 employee records using `head()`.
4. Display last 5 employee records using `tail()`.
5. Use `describe()` for numeric summary.
6. Use `corr()` to find relationships among experience, salary, rating, attendance, and completed projects.
7. Rename `MonthlySalary` as `Salary` and `PerformanceRating` as `Rating` using `rename()`.
8. Check duplicate employee rows using `duplicated()`.
9. Remove duplicate employee rows using `drop_duplicates()`.
10. Fill missing `Salary` values using department-wise median salary.
11. Fill missing `Rating` values using overall median rating.
12. Use `filter()` to display important employee columns.
13. Use `value_counts()` to count employees by city, department, and work mode.
14. Use `sort_values()` to display the highest-paid employees.
15. Use `loc[]` to find senior employees with high ratings.
16. Use `iloc[]` to select a row and column slice by position.
17. Use `query()` to find fast-track bonus candidates.
18. Use `isin()` to select employees from selected departments and cities.
19. Merge employees with department data using `Department`.
20. Merge the result with training data using `EmpID`.
21. Create calculated columns for estimated bonus and training completion flag.
22. Use `groupby()` with multiple aggregations to calculate employee count, average salary, max salary, min salary, average rating, average attendance, total projects, training completion rate, and total estimated bonus by department and manager.
23. Use `reset_index()` after groupby.
24. Concatenate three new employee records with the cleaned employee DataFrame using `concat()`.
25. Create a final department-workmode payroll summary.

## Expected difficulty level

This is a hard-level question because it requires missing value imputation using group-level statistics, multi-table merging, calculated columns, advanced filtering, concatenation, and multiple grouped aggregations.

# Solution 2: Employee Analysis

The following cells solve Question 2 step by step. Run them in order.

## Step 1: Read the Embedded CSV Datasets

The employee, department, and training datasets are loaded from CSV strings.

In [22]:
employees_csv = """EmpID,Name,Department,City,Experience,MonthlySalary,PerformanceRating,AttendancePct,ProjectsCompleted,WorkMode
201,Raj,HR,Ahmedabad,5,45000,4.3,92,6,Hybrid
202,Sneha,IT,Surat,3,52000,4.5,95,8,Remote
203,Ajay,Finance,Rajkot,8,65000,4.7,91,9,Office
204,Nisha,IT,Ahmedabad,4,56000,4.2,89,7,Hybrid
205,Vikas,HR,Vadodara,6,48000,4.4,93,5,Office
206,Anjali,Finance,Surat,7,70000,4.8,96,10,Hybrid
207,Rohan,Marketing,Rajkot,2,39000,4.1,86,4,Remote
208,Kajal,IT,Ahmedabad,5,61000,4.6,94,9,Office
209,Deep,Marketing,Surat,3,42000,4.0,88,5,Hybrid
210,Sonal,Finance,Vadodara,9,76000,4.9,97,11,Office
211,Hardik,IT,Ahmedabad,4,58000,4.4,90,7,Remote
212,Komal,HR,Surat,2,43000,4.2,87,4,Hybrid
213,Neel,Finance,Ahmedabad,6,,4.6,92,8,Office
214,Priya,Marketing,Vadodara,5,50000,,85,6,Remote
215,Aman,IT,Rajkot,7,72000,4.7,96,10,Hybrid
216,Megha,HR,Ahmedabad,8,59000,4.8,94,9,Office
217,Farhan,Marketing,Surat,4,47000,4.3,89,7,Hybrid
218,Leena,Finance,Rajkot,10,82000,4.9,98,12,Office
219,Chirag,IT,Vadodara,6,,4.5,91,8,Remote
220,Ritika,HR,Rajkot,3,44000,4.0,84,5,Hybrid
215,Aman,IT,Rajkot,7,72000,4.7,96,10,Hybrid
"""

departments_csv = """Department,Manager,AnnualBudget,BonusRate
HR,Ms. Desai,1200000,0.08
IT,Mr. Shah,2500000,0.12
Finance,Ms. Rao,2200000,0.15
Marketing,Mr. Mehta,1500000,0.10
"""

training_csv = """EmpID,TrainingProgram,TrainingHours,TrainingCompleted
201,Leadership Basics,12,Yes
202,Cloud Foundations,20,Yes
203,Financial Risk,18,Yes
204,Cloud Foundations,20,No
205,People Analytics,14,Yes
206,Financial Risk,18,Yes
207,Digital Campaigns,16,No
208,Data Engineering,22,Yes
209,Digital Campaigns,16,Yes
210,Audit Excellence,24,Yes
211,Security Basics,12,Yes
212,People Analytics,14,No
213,Financial Risk,18,Yes
214,Brand Strategy,16,No
215,AI Productivity,20,Yes
216,Leadership Advanced,18,Yes
217,Digital Campaigns,16,Yes
218,Audit Excellence,24,Yes
219,Cloud Foundations,20,No
220,People Analytics,14,Yes
"""

employees = pd.read_csv(StringIO(employees_csv))
departments = pd.read_csv(StringIO(departments_csv))
training = pd.read_csv(StringIO(training_csv))

print("Employees shape:", employees.shape)
print("Departments shape:", departments.shape)
print("Training shape:", training.shape)

Employees shape: (21, 10)
Departments shape: (4, 4)
Training shape: (20, 4)


## Step 2: Display First and Last Records

Use `head()` and `tail()` to inspect employee records.

In [23]:
display(employees.head(7))
display(employees.tail(5))

,EmpID,Name,Department,City,Experience,MonthlySalary,PerformanceRating,AttendancePct,ProjectsCompleted,WorkMode
0,201,Raj,HR,Ahmedabad,5,45000.0,4.3,92,6,Hybrid
1,202,Sneha,IT,Surat,3,52000.0,4.5,95,8,Remote
2,203,Ajay,Finance,Rajkot,8,65000.0,4.7,91,9,Office
3,204,Nisha,IT,Ahmedabad,4,56000.0,4.2,89,7,Hybrid
4,205,Vikas,HR,Vadodara,6,48000.0,4.4,93,5,Office
5,206,Anjali,Finance,Surat,7,70000.0,4.8,96,10,Hybrid
6,207,Rohan,Marketing,Rajkot,2,39000.0,4.1,86,4,Remote


,EmpID,Name,Department,City,Experience,MonthlySalary,PerformanceRating,AttendancePct,ProjectsCompleted,WorkMode
16,217,Farhan,Marketing,Surat,4,47000.0,4.3,89,7,Hybrid
17,218,Leena,Finance,Rajkot,10,82000.0,4.9,98,12,Office
18,219,Chirag,IT,Vadodara,6,NaN,4.5,91,8,Remote
19,220,Ritika,HR,Rajkot,3,44000.0,4.0,84,5,Hybrid
20,215,Aman,IT,Rajkot,7,72000.0,4.7,96,10,Hybrid


## Step 3: Statistical Summary

`describe()` summarizes salary, experience, rating, attendance, and projects.

In [24]:
employees.describe()

,EmpID,Experience,MonthlySalary,PerformanceRating,AttendancePct,ProjectsCompleted
count,21.000000,21.000000,19.000000,20.00000,21.000000,21.000000
mean,210.714286,5.428571,56894.736842,4.48000,91.571429,7.619048
std,5.849298,2.270934,12918.095563,0.28764,4.105745,2.312492
min,201.000000,2.000000,39000.000000,4.00000,84.000000,4.000000
25%,206.000000,4.000000,46000.000000,4.27500,89.000000,6.000000
50%,211.000000,5.000000,56000.000000,4.50000,92.000000,8.000000
75%,215.000000,7.000000,67500.000000,4.70000,95.000000,9.000000
max,220.000000,10.000000,82000.000000,4.90000,98.000000,12.000000


## Step 4: Correlation Analysis

`corr()` helps identify relationships among numeric employee variables.

In [25]:
employees.corr(numeric_only=True)

,EmpID,Experience,MonthlySalary,PerformanceRating,AttendancePct,ProjectsCompleted
EmpID,1.000000,0.182828,0.192436,0.105150,-0.094878,0.172677
Experience,0.182828,1.000000,0.866718,0.894546,0.749995,0.851455
MonthlySalary,0.192436,0.866718,1.000000,0.901821,0.813626,0.959729
PerformanceRating,0.105150,0.894546,0.901821,1.000000,0.899264,0.920418
AttendancePct,-0.094878,0.749995,0.813626,0.899264,1.000000,0.850868
ProjectsCompleted,0.172677,0.851455,0.959729,0.920418,0.850868,1.000000


## Step 5: Rename Columns

Column names are simplified for cleaner analysis.

In [26]:
employees_renamed = employees.rename(columns={"MonthlySalary": "Salary", "PerformanceRating": "Rating"})
employees_renamed.head()

,EmpID,Name,Department,City,Experience,Salary,Rating,AttendancePct,ProjectsCompleted,WorkMode
0,201,Raj,HR,Ahmedabad,5,45000.0,4.3,92,6,Hybrid
1,202,Sneha,IT,Surat,3,52000.0,4.5,95,8,Remote
2,203,Ajay,Finance,Rajkot,8,65000.0,4.7,91,9,Office
3,204,Nisha,IT,Ahmedabad,4,56000.0,4.2,89,7,Hybrid
4,205,Vikas,HR,Vadodara,6,48000.0,4.4,93,5,Office


## Step 6: Detect Duplicate Rows

`duplicated()` checks if the same employee row appears more than once.

In [27]:
employees_renamed.duplicated().value_counts()

False    20
True      1
Name: count, dtype: int64

## Step 7: Remove Duplicate Rows

`drop_duplicates()` removes repeated employee records.

In [28]:
employees_clean = employees_renamed.drop_duplicates().copy()
print("Before removing duplicates:", employees_renamed.shape)
print("After removing duplicates:", employees_clean.shape)

Before removing duplicates: (21, 10)
After removing duplicates: (20, 10)


## Step 8: Fill Missing Values

Missing salaries are filled using department-wise median salary. Missing ratings are filled with the overall median rating.

In [29]:
employees_clean["Salary"] = employees_clean["Salary"].fillna(employees_clean.groupby("Department")["Salary"].transform("median"))
employees_clean["Rating"] = employees_clean["Rating"].fillna(employees_clean["Rating"].median())
employees_clean.loc[employees_clean["EmpID"].isin([213, 214, 219]), ["EmpID", "Name", "Department", "Salary", "Rating"]]

,EmpID,Name,Department,Salary,Rating
12,213,Neel,Finance,73000.0,4.6
13,214,Priya,Marketing,50000.0,4.5
18,219,Chirag,IT,58000.0,4.5


## Step 9: Filter Important Columns

`filter()` selects only required columns by name.

In [30]:
employees_clean.filter(items=["EmpID", "Name", "Department", "City", "Salary", "Rating"]).head(10)

,EmpID,Name,Department,City,Salary,Rating
0,201,Raj,HR,Ahmedabad,45000.0,4.3
1,202,Sneha,IT,Surat,52000.0,4.5
2,203,Ajay,Finance,Rajkot,65000.0,4.7
3,204,Nisha,IT,Ahmedabad,56000.0,4.2
4,205,Vikas,HR,Vadodara,48000.0,4.4
5,206,Anjali,Finance,Surat,70000.0,4.8
6,207,Rohan,Marketing,Rajkot,39000.0,4.1
7,208,Kajal,IT,Ahmedabad,61000.0,4.6
8,209,Deep,Marketing,Surat,42000.0,4.0
9,210,Sonal,Finance,Vadodara,76000.0,4.9


## Step 10: Count Employee Distributions

`value_counts()` shows the number of employees by city, department, and work mode.

In [31]:
display(employees_clean["City"].value_counts())
display(employees_clean["Department"].value_counts())
display(employees_clean["WorkMode"].value_counts())

City
Ahmedabad    6
Surat        5
Rajkot       5
Vadodara     4
Name: count, dtype: int64

Department
IT           6
HR           5
Finance      5
Marketing    4
Name: count, dtype: int64

WorkMode
Hybrid    8
Office    7
Remote    5
Name: count, dtype: int64

## Step 11: Sort by Salary

`sort_values()` displays the highest-paid employees first.

In [32]:
employees_clean.sort_values(by="Salary", ascending=False).head(8)

,EmpID,Name,Department,City,Experience,Salary,Rating,AttendancePct,ProjectsCompleted,WorkMode
17,218,Leena,Finance,Rajkot,10,82000.0,4.9,98,12,Office
9,210,Sonal,Finance,Vadodara,9,76000.0,4.9,97,11,Office
12,213,Neel,Finance,Ahmedabad,6,73000.0,4.6,92,8,Office
14,215,Aman,IT,Rajkot,7,72000.0,4.7,96,10,Hybrid
5,206,Anjali,Finance,Surat,7,70000.0,4.8,96,10,Hybrid
2,203,Ajay,Finance,Rajkot,8,65000.0,4.7,91,9,Office
7,208,Kajal,IT,Ahmedabad,5,61000.0,4.6,94,9,Office
15,216,Megha,HR,Ahmedabad,8,59000.0,4.8,94,9,Office


## Step 12: Use loc for Senior High Performers

`loc[]` selects employees by conditions and displays selected columns.

In [33]:
employees_clean.loc[(employees_clean["Experience"] >= 6) & (employees_clean["Rating"] >= 4.6), ["EmpID", "Name", "Department", "Experience", "Salary", "Rating", "ProjectsCompleted"]]

,EmpID,Name,Department,Experience,Salary,Rating,ProjectsCompleted
2,203,Ajay,Finance,8,65000.0,4.7,9
5,206,Anjali,Finance,7,70000.0,4.8,10
9,210,Sonal,Finance,9,76000.0,4.9,11
12,213,Neel,Finance,6,73000.0,4.6,8
14,215,Aman,IT,7,72000.0,4.7,10
15,216,Megha,HR,8,59000.0,4.8,9
17,218,Leena,Finance,10,82000.0,4.9,12


## Step 13: Use iloc for Position-Based Selection

`iloc[]` selects employee data using row and column positions.

In [34]:
employees_clean.iloc[4:12, 0:8]

,EmpID,Name,Department,City,Experience,Salary,Rating,AttendancePct
4,205,Vikas,HR,Vadodara,6,48000.0,4.4,93
5,206,Anjali,Finance,Surat,7,70000.0,4.8,96
6,207,Rohan,Marketing,Rajkot,2,39000.0,4.1,86
7,208,Kajal,IT,Ahmedabad,5,61000.0,4.6,94
8,209,Deep,Marketing,Surat,3,42000.0,4.0,88
9,210,Sonal,Finance,Vadodara,9,76000.0,4.9,97
10,211,Hardik,IT,Ahmedabad,4,58000.0,4.4,90
11,212,Komal,HR,Surat,2,43000.0,4.2,87


## Step 14: Use query for Bonus Candidates

`query()` is used to find employees who satisfy bonus eligibility rules.

In [35]:
employees_clean.query("Rating >= 4.6 and AttendancePct >= 92 and ProjectsCompleted >= 8")[["EmpID", "Name", "Department", "Salary", "Rating", "AttendancePct", "ProjectsCompleted"]]

,EmpID,Name,Department,Salary,Rating,AttendancePct,ProjectsCompleted
5,206,Anjali,Finance,70000.0,4.8,96,10
7,208,Kajal,IT,61000.0,4.6,94,9
9,210,Sonal,Finance,76000.0,4.9,97,11
12,213,Neel,Finance,73000.0,4.6,92,8
14,215,Aman,IT,72000.0,4.7,96,10
15,216,Megha,HR,59000.0,4.8,94,9
17,218,Leena,Finance,82000.0,4.9,98,12


## Step 15: Use isin for Selected Groups

`isin()` selects employees whose department and city match allowed values.

In [36]:
employees_clean[employees_clean["Department"].isin(["IT", "Finance"]) & employees_clean["City"].isin(["Ahmedabad", "Rajkot", "Vadodara"])][["EmpID", "Name", "Department", "City", "Salary", "Rating"]]

,EmpID,Name,Department,City,Salary,Rating
2,203,Ajay,Finance,Rajkot,65000.0,4.7
3,204,Nisha,IT,Ahmedabad,56000.0,4.2
7,208,Kajal,IT,Ahmedabad,61000.0,4.6
9,210,Sonal,Finance,Vadodara,76000.0,4.9
10,211,Hardik,IT,Ahmedabad,58000.0,4.4
12,213,Neel,Finance,Ahmedabad,73000.0,4.6
14,215,Aman,IT,Rajkot,72000.0,4.7
17,218,Leena,Finance,Rajkot,82000.0,4.9
18,219,Chirag,IT,Vadodara,58000.0,4.5


## Step 16: Merge Employee, Department, and Training Data

The employee dataset is enriched with manager, budget, bonus rate, and training details.

In [37]:
employee_department = employees_clean.merge(departments, on="Department", how="left")
full_employees = employee_department.merge(training, on="EmpID", how="left")
full_employees[["EmpID", "Name", "Department", "Manager", "TrainingProgram", "TrainingCompleted", "Salary", "Rating"]].head(10)

,EmpID,Name,Department,Manager,TrainingProgram,TrainingCompleted,Salary,Rating
0,201,Raj,HR,Ms. Desai,Leadership Basics,Yes,45000.0,4.3
1,202,Sneha,IT,Mr. Shah,Cloud Foundations,Yes,52000.0,4.5
2,203,Ajay,Finance,Ms. Rao,Financial Risk,Yes,65000.0,4.7
3,204,Nisha,IT,Mr. Shah,Cloud Foundations,No,56000.0,4.2
4,205,Vikas,HR,Ms. Desai,People Analytics,Yes,48000.0,4.4
5,206,Anjali,Finance,Ms. Rao,Financial Risk,Yes,70000.0,4.8
6,207,Rohan,Marketing,Mr. Mehta,Digital Campaigns,No,39000.0,4.1
7,208,Kajal,IT,Mr. Shah,Data Engineering,Yes,61000.0,4.6
8,209,Deep,Marketing,Mr. Mehta,Digital Campaigns,Yes,42000.0,4.0
9,210,Sonal,Finance,Ms. Rao,Audit Excellence,Yes,76000.0,4.9


## Step 17: Create Calculated Columns

Estimated bonus is calculated from salary and bonus rate. Training completion is converted into a numeric flag.

In [38]:
full_employees["EstimatedBonus"] = full_employees["Salary"] * full_employees["BonusRate"]
full_employees["TrainingCompletedFlag"] = full_employees["TrainingCompleted"].map({"Yes": 1, "No": 0})
full_employees[["EmpID", "Name", "Department", "Salary", "BonusRate", "EstimatedBonus", "TrainingCompletedFlag"]].head()

,EmpID,Name,Department,Salary,BonusRate,EstimatedBonus,TrainingCompletedFlag
0,201,Raj,HR,45000.0,0.08,3600.0,1
1,202,Sneha,IT,52000.0,0.12,6240.0,1
2,203,Ajay,Finance,65000.0,0.15,9750.0,1
3,204,Nisha,IT,56000.0,0.12,6720.0,0
4,205,Vikas,HR,48000.0,0.08,3840.0,1


## Step 18: Groupby with Multiple Aggregations

This report summarizes salary, performance, attendance, projects, training, and bonus by department and manager.

In [39]:
department_summary = (
    full_employees
    .groupby(["Department", "Manager"])
    .agg(
        employee_count=("EmpID", "count"),
        avg_salary=("Salary", "mean"),
        max_salary=("Salary", "max"),
        min_salary=("Salary", "min"),
        avg_rating=("Rating", "mean"),
        avg_attendance=("AttendancePct", "mean"),
        total_projects=("ProjectsCompleted", "sum"),
        training_completion_rate=("TrainingCompletedFlag", "mean"),
        total_estimated_bonus=("EstimatedBonus", "sum")
    )
    .reset_index()
)
department_summary.sort_values(by="avg_salary", ascending=False)

,Department,Manager,employee_count,avg_salary,max_salary,min_salary,avg_rating,avg_attendance,total_projects,training_completion_rate,total_estimated_bonus
0,Finance,Ms. Rao,5,73200.0,82000.0,65000.0,4.780000,94.8,50,1.000000,54900.0
2,IT,Mr. Shah,6,59500.0,72000.0,52000.0,4.483333,92.5,49,0.666667,42840.0
1,HR,Ms. Desai,5,47800.0,59000.0,43000.0,4.340000,90.0,29,0.800000,19120.0
3,Marketing,Mr. Mehta,4,44500.0,50000.0,39000.0,4.225000,87.0,22,0.500000,17800.0


## Step 19: Concatenate New Employee Records

`concat()` adds new employee rows to the cleaned employee DataFrame.

In [40]:
new_employees_csv = """EmpID,Name,Department,City,Experience,Salary,Rating,AttendancePct,ProjectsCompleted,WorkMode
221,Varun,IT,Surat,5,64000,4.5,93,8,Hybrid
222,Jinal,Finance,Ahmedabad,4,60000,4.4,90,7,Office
223,Parth,Marketing,Rajkot,3,45500,4.2,88,6,Remote
"""
new_employees = pd.read_csv(StringIO(new_employees_csv))
employees_final = pd.concat([employees_clean, new_employees], ignore_index=True)
print("Final employees shape:", employees_final.shape)
employees_final.tail(6)

Final employees shape: (23, 10)


,EmpID,Name,Department,City,Experience,Salary,Rating,AttendancePct,ProjectsCompleted,WorkMode
17,218,Leena,Finance,Rajkot,10,82000.0,4.9,98,12,Office
18,219,Chirag,IT,Vadodara,6,58000.0,4.5,91,8,Remote
19,220,Ritika,HR,Rajkot,3,44000.0,4.0,84,5,Hybrid
20,221,Varun,IT,Surat,5,64000.0,4.5,93,8,Hybrid
21,222,Jinal,Finance,Ahmedabad,4,60000.0,4.4,90,7,Office
22,223,Parth,Marketing,Rajkot,3,45500.0,4.2,88,6,Remote


## Step 20: Final Department and Work Mode Summary

The final summary estimates payroll cost and performance after adding new employees.

In [41]:
employees_final_enriched = employees_final.merge(departments, on="Department", how="left")
employees_final_enriched["EstimatedBonus"] = employees_final_enriched["Salary"] * employees_final_enriched["BonusRate"]
employees_final_enriched["EstimatedMonthlyCost"] = employees_final_enriched["Salary"] + employees_final_enriched["EstimatedBonus"]

final_payroll_summary = (
    employees_final_enriched
    .groupby(["Department", "WorkMode"])
    .agg(
        employees=("EmpID", "count"),
        total_salary=("Salary", "sum"),
        avg_salary=("Salary", "mean"),
        avg_rating=("Rating", "mean"),
        avg_experience=("Experience", "mean"),
        total_projects=("ProjectsCompleted", "sum"),
        estimated_monthly_cost=("EstimatedMonthlyCost", "sum")
    )
    .reset_index()
    .sort_values(by="estimated_monthly_cost", ascending=False)
)
final_payroll_summary

,Department,WorkMode,employees,total_salary,avg_salary,avg_rating,avg_experience,total_projects,estimated_monthly_cost
1,Finance,Office,5,356000.0,71200.000000,4.700000,7.400000,47,409400.0
4,IT,Hybrid,3,192000.0,64000.000000,4.466667,5.333333,25,215040.0
6,IT,Remote,3,168000.0,56000.000000,4.466667,4.333333,23,188160.0
8,Marketing,Remote,3,134500.0,44833.333333,4.266667,3.333333,16,147950.0
2,HR,Hybrid,3,132000.0,44000.000000,4.166667,3.333333,15,142560.0
3,HR,Office,2,107000.0,53500.000000,4.600000,7.000000,14,115560.0
7,Marketing,Hybrid,2,89000.0,44500.000000,4.150000,3.500000,12,97900.0
0,Finance,Hybrid,1,70000.0,70000.000000,4.800000,7.000000,10,80500.0
5,IT,Office,1,61000.0,61000.000000,4.600000,5.000000,9,68320.0
